<a href="https://colab.research.google.com/github/chakitudomsinsakorn-crypto/sdxl1/blob/main/stable_diffusion_interactive_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 👇 Installing dependencies { display-mode: "form" }
#@markdown ---
#@markdown Make sure to select **GPU** as the runtime type:<br/>
#@markdown *Runtime->Change Runtime Type->Under Hardware accelerator, select GPU*
#@markdown
#@markdown ---

!pip -q install torch diffusers transformers accelerate scipy safetensors xformers mediapy ipywidgets==7.7.1

# Stable Diffusion Interactive Notebook 📓 🤖

A widgets-based interactive notebook for Google Colab that lets users generate AI images from prompts (Text2Image) using [Stable Diffusion (by Stability AI, Runway & CompVis)](https://en.wikipedia.org/wiki/Stable_Diffusion).

This notebook aims to be an alternative to WebUIs while offering a simple and lightweight GUI for anyone to get started with Stable Diffusion.

Uses Stable Diffusion, [HuggingFace](https://huggingface.co/) Diffusers and [Jupyter widgets](https://github.com/jupyter-widgets/ipywidgets).

<br/>

Made with ❤️ by redromnon

[GitHub](https://github.com/redromnon/stable-diffusion-interactive-notebook)

In [ ]:
!pip install mediapy


In [ ]:
url = "https://" + "download.pytorch.org" + "/whl/cu130"
!pip install -U mslk --index-url {url}


In [ ]:
#@title 👇 Selecting Model { form-width: "20%", display-mode: "form" }
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler, DDIMScheduler, EulerDiscreteScheduler, UniPCMultistepScheduler
from diffusers.models import AutoencoderKL
import torch
import ipywidgets as widgets
from google.colab import output

output.enable_custom_widget_manager()
pipe = None

select_model = widgets.Dropdown(
    options=[
        ("Animagine XL 4.0", "cagliostrolab/animagine-xl-4.0"),
        ("pony","John6666/autismmix-sdxl-autismmix-pony-sdxl"),
        ("Stable Diffusion XL Base 1.0", "stabilityai/stable-diffusion-xl-base-1.0"),
        ("SSD-1B (Fast SDXL)", "segmind/SSD-1B"),
    ],

    value="cagliostrolab/animagine-xl-4.0",

    description="Select Model:"
)

select_sampler = widgets.Dropdown(
    options=[
        "EulerAncestralDiscreteScheduler",
        "EulerDiscreteScheduler",
        "UniPCMultistepScheduler",
        "DDIMScheduler"
    ],
    description="Select Scheduler:"
)
select_sampler.style.description_width = "auto"

out = widgets.Output()
apply_btn = widgets.Button(description="Apply", button_style="info")

def get_scheduler(name):
    schedulers = {
        "EulerAncestralDiscreteScheduler": EulerAncestralDiscreteScheduler,
        "DDIMScheduler": DDIMScheduler,
        "EulerDiscreteScheduler": EulerDiscreteScheduler,
        "UniPCMultistepScheduler": UniPCMultistepScheduler,
    }
    return schedulers[name].from_pretrained(select_model.value, subfolder="scheduler")

def pipeline(p):
    global pipe
    out.clear_output()
    apply_btn.disabled = True
    with out:
        print("Running, please wait...")
        try:
            vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
            pipe = StableDiffusionXLPipeline.from_pretrained(
                select_model.value,
                scheduler=get_scheduler(select_sampler.value),
                torch_dtype=torch.float16,
                vae=vae
            ).to("cuda")
            print("Finished!")
        except Exception as e:
            print(f"Error: {e}")
    apply_btn.disabled = False

apply_btn.on_click(pipeline)

widgets.VBox([
    widgets.HTML(value="<h2>Configure Pipeline</h2>"),
    select_model, select_sampler, apply_btn, out
])

In [ ]:
#@title 👇 Generating Images { form-width: "20%", display-mode: "form" }
#@markdown ---
#@markdown - **Prompt** - Description of the image
#@markdown - **Negative Prompt** - Things you don't want to see or ignore in the image
#@markdown - **Steps** - Number of denoising steps. Higher steps may lead to better results but takes longer time to generate the image. Default is `30`.
#@markdown - **CFG** - Guidance scale ranging from `0` to `20`. Lower values allow the AI to be more creative and less strict at following the prompt. Default is `7.5`.
#@markdown - **Seed** - A random value that controls image generation. The same seed and prompt produce the same images. Set `-1` for using random seed values.
#@markdown ---
import ipywidgets as widgets, mediapy, random
import IPython.display


#PARAMETER WIDGETS
width = "300px"

prompt = widgets.Textarea(
    value="",
    placeholder="Enter prompt",
    #description="Prompt:",
    rows=5,
    layout=widgets.Layout(width="600px")
)

neg_prompt = widgets.Textarea(
    value="",
    placeholder="Enter negative prompt",
    #description="Negative Prompt:",
    rows=5,
    layout=widgets.Layout(width="600px")
)

num_images = widgets.IntText(
    value=1,
    description="Images:",
    layout=widgets.Layout(width=width),
)

steps = widgets.IntText(
    value=30,
    description="Steps:",
    layout=widgets.Layout(width=width)
)

CFG = widgets.FloatText(
    value=7.5,
    description="CFG:",
    layout=widgets.Layout(width=width)
)

img_height = widgets.Dropdown(
    options=[('1024px', 1024), ('1216px', 1216), ('832px', 832),('768px', 768)],
    value=1024,
    description="Height:",
    layout=widgets.Layout(width=width)
)

img_width = widgets.Dropdown(
    options=[('1024px', 1024),('1216px', 1216), ('832px', 832), ('768px', 768)],
    value=1024,
    description="Width:",
    layout=widgets.Layout(width=width)
)

random_seed = widgets.IntText(
    value=-1,
    description="Seed:",
    layout=widgets.Layout(width=width),
    disabled=False
)

generate = widgets.Button(
    description="Generate",
    disabled=False,
    button_style="primary"
)

display_imgs = widgets.Output()


#RUN
def generate_img(i):

  #Clear output
  display_imgs.clear_output()
  generate.disabled = True

  #Calculate seed
  seed = random.randint(0, 2147483647) if random_seed.value == -1 else random_seed.value

  with display_imgs:

    print("Running...")

    images = pipe(
      prompt.value,
      height = img_height.value,
      width = img_width.value,
      num_inference_steps = steps.value,
      guidance_scale = CFG.value,
      num_images_per_prompt = num_images.value,
      negative_prompt = neg_prompt.value,
      generator = torch.Generator("cuda").manual_seed(seed),
    ).images
    mediapy.show_images(images)

    print(f"Seed:\n{seed}")

  generate.disabled = False

#Display
generate.on_click(generate_img)

widgets.VBox(
    [
      widgets.AppLayout(
        header=widgets.HTML(
            value="<h2>Stable Diffusion</h2>",
        ),
        left_sidebar=widgets.VBox(
            [num_images, steps, CFG, img_height, img_width, random_seed]
        ),
        center=widgets.VBox(
            [prompt, neg_prompt, generate]
        ),
        right_sidebar=None,
        footer=None
      ),
      display_imgs
    ]
)

# ส่วนใหม่

In [ ]:
!pip install -q gradio


In [ ]:
import gradio as gr
import torch
import random

def generate_images(prompt, negative_prompt, width, height, steps, cfg_scale, num_images, seed):
    if seed == -1:
        seed = random.randint(0, 2147483647)

    generator = torch.Generator("cuda").manual_seed(int(seed))

    images = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=int(width),
        height=int(height),
        num_inference_steps=int(steps),
        guidance_scale=float(cfg_scale),
        num_images_per_prompt=int(num_images),
        generator=generator
    ).images

    return images, f"Seed: {seed}"

size_list = [768, 832, 1024, 1216, 1344]

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## Animagine XL 4.0 Studio")

    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", lines=4, placeholder="1girl, ayase momo, dandadan...")
            neg_prompt = gr.Textbox(
                label="Negative Prompt",
                lines=2,
                value="lowres, bad anatomy, bad hands, text, error, missing finger, extra digits, fewer digits, cropped, worst quality, low quality, normal quality, jpeg artifacts, signature, watermark, username, blurry, artist name"
            )

            with gr.Row():
                width = gr.Dropdown(label="Width", choices=size_list, value=832)
                height = gr.Dropdown(label="Height", choices=size_list, value=1216)

            with gr.Row():
                steps = gr.Slider(label="Steps", minimum=15, maximum=50, step=1, value=28)
                cfg = gr.Slider(label="CFG Scale", minimum=1.0, maximum=12.0, step=0.5, value=5.0)

            with gr.Row():
                num_images = gr.Slider(label="Batch Size", minimum=1, maximum=4, step=1, value=2)
                seed = gr.Number(label="Seed (-1 = Random)", value=-1, precision=0)

            btn = gr.Button("Generate", variant="primary")

        with gr.Column(scale=1):
            gallery = gr.Gallery(label="Gallery", columns=2, height="auto")
            seed_info = gr.Textbox(label="Info", interactive=False)

    btn.click(
        fn=generate_images,
        inputs=[prompt, neg_prompt, width, height, steps, cfg, num_images, seed],
        outputs=[gallery, seed_info]
    )

demo.launch(share=True, inline=True)